# Phase 1 - Load, Explore & Clean

In [1]:
import pandas as pd
import numpy as np

We import **pandas** for working with tables (DataFrames) and **numpy** for math operations. These are the two most common libraries in any data cleaning project.

In [2]:
titanic = pd.read_csv(
    "data/raw/train.csv",
    quotechar='"',
    skipinitialspace=True
)
print(titanic.shape)
print(titanic.head())
print(titanic.info())
print(titanic.describe())

(891, 12)
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

  Name                                                                                  \
0  Braund, Mr. Owen Harris                       ...                                     
1  Cumings, Mrs. John Bradley (Florence Briggs Th...                                     
2  Heikkinen, Miss. Laina                        ...                                     
3  Futrelle, Mrs. Jacques Heath (Lily May Peel)  ...                                     
4  Allen, Mr. William Henry                      ...                                     

   Sex     Age    SibSp  Parch  Ticket              Fare      Cabin            \
0  male     22.0      1      0  A/5 21171             7.2500              NaN   
1  female   38.0      1      0  PC 17599             71.2833  C85     

We load the Titanic CSV with `quotechar` and `skipinitialspace=True` to handle any extra spaces or quoting issues in the file. Then we run 4 commands to understand the data:
- **shape** — gives us 891 rows and 12 columns
- **head()** — shows first 5 rows so we can see what the data looks like
- **info()** — shows data types and how many non-null values per column (this is where we spot missing data: Age has 177 missing, Cabin has 687 missing, Embarked has 2 missing)
- **describe()** — gives stats like mean, min, max for numeric columns (helps spot outliers like max Fare = 512 vs median of 14)

In [3]:
titanic.columns = titanic.columns.str.strip()
titanic["Sex"] = titanic["Sex"].str.strip()
titanic["Embarked"] = titanic["Embarked"].str.strip()
titanic["Embarked"] = titanic["Embarked"].fillna("S")

Three quick fixes here:
- **str.strip()** removes any extra spaces from column names, Sex, and Embarked (prevents KeyError or mismatched mappings)
- **fillna("S")** fills the 2 missing Embarked values with "S" (Southampton), which is the most common port (72% of passengers)
- We strip Sex and Embarked early so the grouped Age fill in the next step works correctly

In [4]:
print(titanic['Age'].dtype)

float64


Checking what type Age is stored as. If it's `object` (text) instead of a number, we can't calculate the median.

In [5]:
titanic['Age'] = pd.to_numeric(titanic['Age'], errors='coerce')

Converts Age to a proper number type. `errors='coerce'` means if any value can't be converted (like a typo), it becomes NaN instead of crashing.

In [6]:
print(titanic['Age'].dtype)
print(titanic['Age'].isnull().sum())

float64
177


Verifying the conversion worked. Age should now be `float64` and we can see how many null values we need to fill.

In [7]:
print(titanic.groupby(["Pclass", "Sex"])["Age"].median())

Pclass  Sex   
1       female    35.0
        male      40.0
2       female    28.0
        male      30.0
3       female    21.5
        male      25.0
Name: Age, dtype: float64


Looking at the median age for each class + gender group. A 1st class female has a different typical age than a 3rd class male, so filling by group is more accurate than one overall median.

In [8]:
titanic["Age"] = titanic.groupby(["Pclass", "Sex"])["Age"].transform(
    lambda x: x.fillna(x.median())
)

The smart fill: for each group (e.g. 1st class males), replaces missing ages with that group's median. `transform()` keeps the same shape as the original column so it fits back in.

In [9]:
titanic["Age"] = titanic["Age"].fillna(titanic["Age"].median())

Safety net: if any group had ALL ages missing, those rows would still be null after the group fill. This catches any leftovers using the overall median.

In [10]:
print(titanic['Age'].isnull().sum())

0


Final Age check — should print **0**. No more missing ages.

In [11]:
titanic = titanic.drop(columns=["Cabin", "Ticket"])

Dropping two columns that won't help predict survival:
- **Cabin** — 77% missing, too much to fill
- **Ticket** — random codes with no clear pattern

We keep Name for now so that each row stays unique during duplicate detection.

In [12]:
print("Duplicates before:", titanic.duplicated().sum())
titanic = titanic.drop_duplicates()
print("Duplicates after:", titanic.duplicated().sum())
print("Shape:", titanic.shape)

Duplicates before: 0
Duplicates after: 0
Shape: (891, 10)


Checking for duplicate rows. With Name still in the dataset, each passenger is unique so we expect 0 duplicates. This confirms our data has no accidental duplicate entries.

In [13]:
print(titanic['Fare'].describe())

count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64


Looking at Fare stats before handling outliers. The big gap between the 75th percentile (31) and the max (512) tells us there are extreme values.

In [14]:
Q1 = titanic['Fare'].quantile(0.25)
Q3 = titanic['Fare'].quantile(0.75)

IQR = Q3 - Q1
print("IQR:", IQR)

Upper_limit = Q3 + 1.5 * IQR
print("Upper limit:", Upper_limit)

IQR: 23.0896
Upper limit: 65.6344


Using the **IQR method** to calculate outlier boundaries. IQR = Q3 - Q1 (the middle 50% of data). Anything above Q3 + 1.5×IQR is an outlier. This is the same math behind boxplot whiskers.

In [15]:
titanic['Fare'] = titanic['Fare'].clip(upper=Upper_limit)
print(titanic['Fare'].describe())

count    891.000000
mean      24.046813
std       20.481625
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max       65.634400
Name: Fare, dtype: float64


Instead of deleting outlier rows (which loses data), we **clip** them: any fare above the upper limit gets capped down to it. We only clip the upper end since fares can't be negative. This keeps all 891 passengers in the dataset while removing extreme values.

In [16]:
titanic = titanic.drop(columns=["Name", "PassengerId"])
print(titanic.dtypes)

Survived      int64
Pclass        int64
Sex          object
Age         float64
SibSp         int64
Parch         int64
Fare        float64
Embarked     object
dtype: object


Now we drop Name and PassengerId — Name was kept for duplicate checking, and PassengerId is just a row number. Sex and Embarked stay as text so that Phase 2 (Feature Engineering) can demonstrate proper encoding techniques.

In [17]:
print(titanic.shape)
print(titanic.head())

(891, 8)
   Survived  Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0         0       3    male  22.0      1      0   7.2500        S
1         1       1  female  38.0      1      0  65.6344        C
2         1       3  female  26.0      0      0   7.9250        S
3         1       1  female  35.0      1      0  53.1000        S
4         0       3    male  35.0      0      0   8.0500        S


Final shape check. We should have 891 rows and 8 columns: Survived, Pclass, Sex, Age, SibSp, Parch, Fare, Embarked. All missing values filled, outliers clipped, no duplicates. Sex and Embarked are still text — encoding happens in Phase 2.

## clean_data() Function

This function wraps all cleaning steps into one reusable pipeline. Given the raw Titanic CSV, it returns a fully cleaned DataFrame. Steps: fix column names → fix data types → fill missing values → drop Cabin/Ticket → remove duplicates → clip outliers → drop Name/PassengerId. Sex and Embarked remain as text for Phase 2 encoding.

In [18]:
def clean_data(df):
    """Full cleaning pipeline for Titanic dataset."""
    df = df.copy()
    
    # 1. Fix column names and strip text columns
    df.columns = df.columns.str.strip()
    df['Sex'] = df['Sex'].str.strip()
    df['Embarked'] = df['Embarked'].str.strip()
    
    # 2. Fix data types
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df['Pclass'] = pd.to_numeric(df['Pclass'], errors='coerce')
    
    # 3. Handle missing values
    df['Embarked'] = df['Embarked'].fillna('S')
    df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(
        lambda x: x.fillna(x.median())
    )
    df['Age'] = df['Age'].fillna(df['Age'].median())
    
    # 4. Drop columns not useful for prediction
    df = df.drop(columns=['Cabin', 'Ticket'])
    
    # 5. Handle duplicates (Name keeps rows unique)
    df = df.drop_duplicates()
    
    # 6. Handle outliers using IQR
    Q1 = df['Fare'].quantile(0.25)
    Q3 = df['Fare'].quantile(0.75)
    IQR = Q3 - Q1
    Upper_limit = Q3 + 1.5 * IQR
    df['Fare'] = df['Fare'].clip(upper=Upper_limit)
    
    # 7. Drop Name and PassengerId (kept for dedup)
    df = df.drop(columns=['Name', 'PassengerId'])
    
    return df

In [19]:
# Test the function
raw = pd.read_csv("data/raw/train.csv", quotechar='"', skipinitialspace=True)
cleaned = clean_data(raw)

# Check 1: No missing values in key columns
assert cleaned[['Survived', 'Age', 'Fare']].isnull().sum().sum() == 0, "Still has missing values!"
print("Check 1 passed: No missing values")

# Check 2: Survived is only 0 or 1
assert cleaned['Survived'].isin([0, 1]).all(), "Survived has invalid values!"
print("Check 2 passed: Survived is only 0 and 1")

# Check 3: Correct number of columns
assert len(cleaned.columns) == 8, f"Expected 8 columns, got {len(cleaned.columns)}"
print("Check 3 passed: 8 columns as expected")

# Check 4: 891 rows preserved
assert len(cleaned) == 891, f"Expected 891 rows, got {len(cleaned)}"
print("Check 4 passed: 891 rows preserved")

# Check 5: Fare outliers clipped
assert cleaned['Fare'].max() <= 65.7, f"Fare max too high: {cleaned['Fare'].max()}"
print("Check 5 passed: Fare outliers clipped")

# Check 6: Sex and Embarked still text
assert cleaned['Sex'].dtype == 'object', "Sex should still be text!"
assert cleaned['Embarked'].dtype == 'object', "Embarked should still be text!"
print("Check 6 passed: Sex and Embarked still text")

print("\nAll checks passed! Shape:", cleaned.shape)

Check 1 passed: No missing values
Check 2 passed: Survived is only 0 and 1
Check 3 passed: 8 columns as expected
Check 4 passed: 891 rows preserved
Check 5 passed: Fare outliers clipped
Check 6 passed: Sex and Embarked still text

All checks passed! Shape: (891, 8)


In [20]:
titanic.to_csv("data/cleaned/cleaned_titanic.csv", index=False)
print("Saved to data/cleaned/cleaned_titanic.csv")

Saved to data/cleaned/cleaned_titanic.csv


Saving the cleaned dataset to a CSV file so we can use it in the next phases (EDA, modeling) without re-running all the cleaning steps.